<a href="https://colab.research.google.com/github/brando710/brando710.github.io/blob/main/Website_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Intall dependencies

In [ ]:
pip install openai requests beautifulsoup4

Make Sure ollama is running

In [ ]:
ollama serve
ollama pull llama3.2

Website Summarizer Code

In [ ]:
# ============================================
# WEBSITE SUMMARIZER USING LLAMA 3.2 LOCALLY
# ============================================

from openai import OpenAI
import requests
from bs4 import BeautifulSoup


# Connect to Ollama running locally
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


# ============================================
# FUNCTION TO EXTRACT WEBSITE TEXT
# ============================================

def get_website_text(url):

    try:
        response = requests.get(url, timeout=10)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unnecessary tags
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        # Prevent sending too much text
        text = text[:5000]

        return text

    except Exception as e:
        return f"Error reading website: {e}"


# ============================================
# GET WEBSITE FROM USER
# ============================================

website_url = input("Enter website URL: ")

website_text = get_website_text(website_url)


# ============================================
# STEP 1: CREATE PROMPTS
# ============================================

system_prompt = """
You are an expert website summarizer.

Your job is to:
- Summarize websites clearly
- Identify the main purpose
- Highlight important information
- Keep summaries concise and professional
"""

user_prompt = f"""
Summarize this website:

{website_text}
"""


# ============================================
# STEP 2: CREATE MESSAGE LIST
# ============================================

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


# ============================================
# STEP 3: CALL LLAMA 3.2 LOCALLY
# ============================================

response = client.chat.completions.create(
    model="llama3.2",
    messages=messages,
    temperature=0.3
)


# ============================================
# STEP 4: PRINT RESULT
# ============================================

print("\n===== WEBSITE SUMMARY =====\n")
print(response.choices[0].message.content)